In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
from torch.ao.quantization.pt2e import prepare_qat_pt2e, convert_pt2e
from torch.ao.quantization.quantizer.qat_config import QATConfig
from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer

import os
import logging
from datetime import datetime

In [ ]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

## Full ResNet18 Traininig

In [ ]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [ ]:
def training_loop(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

In [ ]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

In [ ]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## QAT ResNet Training

In [ ]:
qat_model = resnet18()
qat_model.fc = nn.Linear(qat_model.fc.in_features, 10)

if isinstance(trainset, torchvision.datasets.CIFAR10):
    qat_model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,          
        padding=1,          
        bias=False
    )

    # Remove maxpool (not needed for small inputs)
    qat_model.maxpool = nn.Identity()

In [ ]:
print(qat_model)

#### [DEPRECATED] QAT via torch.ao.quantization.prepare_qat (Eager way, older)

In [ ]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original parameters / submodules
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # local quant/dequant stubs (these use observers when prepared)
        self.quant = aq.QuantStub()
        self.dequant = aq.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize at block entry => convs will be fake-quantized during QAT
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize so addition runs in FP32
        out = self.dequant(out)

        if self.downsample is not None:
            # keep downsample in FP32 by applying it directly on FP32 input
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [ ]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            layer[i] = QuantizableBasicBlock(orig_block)

make_blocks_quantizable(qat_model)

In [ ]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig

qat_model.conv1.qconfig = None
qat_model.fc.qconfig = None

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(qat_model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # set downsample (Sequential) to FP32
            block.downsample.qconfig = None

In [ ]:
qat_model_prepared = torch.ao.quantization.prepare_qat(qat_model)
print(qat_model_prepared)

#### [DEPRECATED] QAT via prepare_qat_fx (New (FX) way. lets you fine-tune per-module quantization in a declarative way)

In [ ]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig
example_input = torch.randn(1, 3, 32, 32) # For checking 

qconfig_dict = {
    "": default_qconfig,  # default for all layers
    "module_name": [("conv1", None), ("fc", None)]  # disable quant for first and last
}

qat_model_prepared = prepare_qat_fx(qat_model, qconfig_dict, example_input)
print(qat_model_prepared)

#### QAT via torch.ao.quantization.pt2e (Recommened)

In [ ]:
example_input = (torch.randn(1, 3, 32, 32),)
exported = torch.export.export(qat_model, example_input).module()

quantizer = XNNPACKQuantizer()

In [ ]:
ternary_qat = QATConfig(
    activation_dtype=torch.quint8,  # activations stay 8-bit
    weight_dtype=torch.qint2,       # 2-bit weights → ternary
    enable_observer=True,
    enable_fake_quant=True,
    observer_kwargs={"quant_min": -1, "quant_max": 1}  # enforce ternary range
)

# --- FP32 config for first & last ---
fp32_qat = QATConfig(
    activation_dtype=None,
    weight_dtype=None,
    enable_fake_quant=False,
    enable_observer=False
)

In [ ]:
quantizer.set_global(ternary_qat)

# Disable quantization for first conv and final fc
quantizer.set_module_name("conv1", fp32_qat)
quantizer.set_module_name("fc", fp32_qat)

In [ ]:
qat_prepared = prepare_qat_pt2e(exported, quantizer)

#### QAT Model traininig

In [ ]:
qat_model_prepared.to(device)
start_of_training_timestamp = training_loop(qat_model_prepared, "resnet18_cifar10_qat", trainloader, testloader, num_epochs=1)

In [ ]:
qat_model_prepared.eval()
qat_model_prepared.to("cpu")
# final_quantized_model = convert_fx(qat_model_prepared)
final_quantized_model = convert_pt2e(qat_prepared)

qt_path = f"./trained_models/resnet18_cifar10_qat_{start_of_training_timestamp}.pth"
torch.save(final_quantized_model.state_dict(), qt_path)

print(f"Quantized model saved as {qt_path}")
print("Quantized file size (MB):", os.path.getsize(qt_path)/(1024**2))

In [ ]:
print(final_quantized_model)

#### QAT Model evaluation

In [ ]:
final_quantized_model.load_state_dict(torch.load("./trained_models/resnet18_cifar10_qat_07.10.2025-15:48:04.pth", map_location="cpu"))
final_quantized_model.eval()

In [ ]:
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to('cpu'), labels.to('cpu')
        outputs = final_quantized_model(inputs)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

test_loss /= total_test
test_acc = 100. * correct_test / total_test

# Log metrics
print(
    f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
)

In [12]:
print(final_quantized_model)

GraphModule(
  (conv1): ConvBnReLU2d(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (maxpool): Identity()
  (layer1): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.0833040103316307, zero_point=0, padding=(1, 1))
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.12375370413064957, zero_point=63, padding=(1, 1))
    )
    (1): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.06330378353595734, zero_point=0, padding=(1, 1))
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.12066991627216339, zero_point=61, padding=(1, 1))
    )
  )
  (layer2): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 128, kernel_size=(3, 3), stride=(2, 2), scale=0.0693